In [1]:
import numpy as np
import matplotlib.pyplot as plt

def sigmoid(z):
    return 1 / (1 + np.exp(-z))

def sigmoid_derivative(z):
    s = sigmoid(z)
    return s * (1 - s)

def relu(z):
    return np.maximum(0, z)

def relu_derivative(z):
    return (z > 0).astype(float)

def binary_cross_entropy(y, y_hat):
    epsilon = 1e-8
    y_hat = np.clip(y_hat, epsilon, 1 - epsilon)
    return -np.mean(y * np.log(y_hat) + (1 - y) * np.log(1 - y_hat))

def compute_accuracy(y, y_hat):
    y_bin = (y_hat >= 0.5).astype(int)
    return np.mean(y_bin == y)

In [2]:
class Optimizer:
    def __init__(self, lr=0.01):
        self.lr = lr

class Momentum(Optimizer):
    def __init__(self, lr=0.01, beta=0.9):
        super().__init__(lr)
        self.beta = beta
        self.v_w, self.v_b = {}, {}

    def update(self, layer_id, w, b, dw, db):
        if layer_id not in self.v_w:
            self.v_w[layer_id] = np.zeros_like(w)
            self.v_b[layer_id] = np.zeros_like(b)

        self.v_w[layer_id] = self.beta * self.v_w[layer_id] + self.lr * dw
        self.v_b[layer_id] = self.beta * self.v_b[layer_id] + self.lr * db
        return w - self.v_w[layer_id], b - self.v_v[layer_id]

In [4]:
class DenseLayer:
    def __init__(self, n_in, n_out, activation='relu'):
        self.W = np.random.uniform(-1, 1, (n_out, n_in))
        self.b = np.zeros((n_out, 1))
        self.activation = activation

    def forward(self, A_prev):
        self.A_prev = A_prev
        self.Z = np.dot(self.W, A_prev) + self.b # Linear transformation [cite: 3556, 3575]
        self.A = relu(self.Z) if self.activation == 'relu' else sigmoid(self.Z)
        return self.A

    def backward(self, dA, lr, layer_id, optimizer):
        slope = relu_derivative(self.Z) if self.activation == 'relu' else sigmoid_derivative(self.Z)
        self.dZ = dA * slope

        dW = np.dot(self.dZ, self.A_prev.T) / self.A_prev.shape[1]
        db = np.mean(self.dZ, axis=1, keepdims=True)

        dA_prev = np.dot(self.W.T, self.dZ)

        self.W -= lr * dW
        self.b -= lr * db
        return dA_prev

In [5]:
class ConvLayer:
    def __init__(self, n_filters, filter_size):
        self.n_filters = n_filters
        self.f = filter_size
        self.filters = np.random.randn(n_filters, filter_size, filter_size) * 0.1

    def forward(self, image):
        self.image = image
        h, w = image.shape
        out_h, out_w = h - self.f + 1, w - self.f + 1
        feature_map = np.zeros((self.n_filters, out_h, out_w)) # Feature Map [cite: 3830]

        for n in range(self.n_filters):
            for i in range(out_h):
                for j in range(out_w):
                    region = image[i:(i+self.f), j:(j+self.f)]
                    feature_map[n, i, j] = np.sum(region * self.filters[n])
        return relu(feature_map)

In [6]:
def report_parameters(architecture):
    """
    Example for 2->3->2->1 Network:
    Layer 1 (2->3): (3 * 2) + 3 = 9
    Layer 2 (3->2): (2 * 3) + 2 = 8
    Layer 3 (2->1): (1 * 2) + 1 = 3
    Total = 20 Parameters
    """
    print("Layer-wise Parameter Count Analysis Complete.")